# Qwen3-ASR 0.6B LoRA Mini Run: 50h Train / 100-Sample Evals

This notebook is a deliberately smaller, separate run derived from `qwen3_asr_0_6b_lora_run.ipynb`.

Differences from the current/full run:
- unique run directory: `/home/MohammadNabulsi/whisper/Runs/qwen3_asr_0_6b_mini_50h_100eval`
- unique cache directory: `/home/MohammadNabulsi/whisper/cache/qwen3_asr_0_6b_mini_50h_100eval`
- fresh manifest rebuild
- deterministic training subset capped at 50 total audio hours
- validation/test/baseline generation capped at 100 samples
- final integrity report rejects empty prediction files


## 1. Config

All run paths and behavior flags live here. The split rules intentionally treat Casablanca `relevant_arabic` and Omnilingual `other_arabic_dialects` as train-only, regardless of filenames like `validation` or `test` inside those roots.


In [ ]:
from pathlib import Path

MODEL_NAME = "Qwen/Qwen3-ASR-0.6B"
MINI_RUN_ID = "qwen3_asr_0_6b_mini_50h_100eval"

RUN_DIR = Path("/home/MohammadNabulsi/whisper/Runs/qwen3_asr_0_6b_mini_50h_100eval")
NOTEBOOK_PATH = RUN_DIR / "qwen3_asr_0_6b_lora_mini_50h_100eval_run.ipynb"
LOG_DIR = RUN_DIR / "logs"
OUTPUT_DIR = RUN_DIR / "checkpoints"
BEST_MODEL_DIR = RUN_DIR / "best"
MANIFEST_DIR = RUN_DIR / "manifests"
CACHE_DIR = Path("/home/MohammadNabulsi/whisper/cache/qwen3_asr_0_6b_mini_50h_100eval")

TRAIN_SOURCE_DIRS = [
    Path("/home/MohammadNabulsi/whisper/casablanca/relevant_arabic"),
    Path("/home/MohammadNabulsi/whisper/omnilingual_selected/other_arabic_dialects"),
]

HELDOUT_VAL_TEST_SOURCE_DIRS = [
    Path("/home/MohammadNabulsi/whisper/omnilingual_selected/apc_north_levantine_all_splits"),
    Path("/home/MohammadNabulsi/whisper/casablanca/levant"),
]

# Optional examples for small tests only, not the full source list.
OMNILINGUAL_ARROW_EXAMPLE = Path("/home/MohammadNabulsi/whisper/omnilingual_selected/apc_north_levantine_all_splits/data-00000-of-00003.arrow")
CASABLANCA_PARQUET_EXAMPLE = Path("/home/MohammadNabulsi/whisper/casablanca/levant/Palestine/test-00001-of-00002.parquet")
QASR_WAV_EXAMPLE = Path("/home/MohammadNabulsi/whisper/QASR/alt/alt/arabic-speech-web/mgb2.1/wav/0A4D5AA5-CA9E-4EB5-99D8-BD6D6DA2C58C.wav")
QASR_XML_EXAMPLE = Path("/home/MohammadNabulsi/whisper/QASR/mgb2.1/release/train_20210109/xml/0A4D5AA5-CA9E-4EB5-99D8-BD6D6DA2C58C.xml")
QASR_ROOT = Path("/home/MohammadNabulsi/whisper/QASR")
INCLUDE_QASR_IN_FULL_MANIFEST = False

SAMPLE_RATE = 16000
MAX_AUDIO_SECONDS = 120.0
MIN_AUDIO_SECONDS = 0.3

SPLIT_SEED = 42
FORCE_RESPLIT = True
VAL_FRACTION_OF_UNSPLIT_HELDOUT = 0.50

MINI_MAX_TRAIN_HOURS = 50.0
MINI_MAX_EVAL_SAMPLES = 100
MINI_MIN_TRAIN_HOURS_REQUIRED = 49.0
MINI_CASABLANCA_MIN_ROWS = 100
MINI_TRAIN_SELECTION_STRATEGY = "longest_first_with_casablanca_seed"
MINI_PRECACHE_BEFORE_TRAIN = True
MINI_CACHE_WAIT_POLL_SECONDS = 30

MIX_TRAIN_SOURCES = True
SOURCE_SAMPLING_STRATEGY = "round_robin"  # or "temperature"
SOURCE_TEMPERATURE = 0.7
BALANCE_SOURCES_IN_BATCH = True

TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-4
NUM_TRAIN_EPOCHS = 1
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

CACHE_NUM_WORKERS = 4
TRAIN_DATALOADER_NUM_WORKERS = 4
CACHE_SHARD_SIZE = 250
PREFETCH_FACTOR = 2
PERSISTENT_WORKERS = True

FORCE_REBUILD_MANIFEST = False
FORCE_REBUILD_CACHE = False
RUN_SMOKE_TEST = False
SMOKE_TEST_ONLY = False
RUN_SMALL_REAL_SAMPLE_TEST = False
DEBUG_STRICT = False

USE_BF16 = True
USE_FP16 = False
RESUME_FROM_CHECKPOINT = False

# Text normalization policy: remove dataset artifacts, not Arabic spelling.
NORMALIZE_ARABIC = True
REMOVE_ASR_TAGS = True
REMOVE_TATWEEL = True
REMOVE_DIACRITICS = True
NORMALIZE_ALEF = False
NORMALIZE_YAA = False
NORMALIZE_TAA_MARBUTA = False
REMOVE_PUNCTUATION = False
COLLAPSE_REPEATED_PUNCT = True
MAP_WESTERN_TO_ARABIC_PUNCT = False
DROP_EMPTY_AFTER_NORMALIZATION = True

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = "auto"

MAX_OMNILINGUAL_SAMPLES_DEBUG = 5
MAX_CASABLANCA_SAMPLES_DEBUG = 5
MAX_QASR_SEGMENTS_DEBUG = 5

PREDICTION_DIR = RUN_DIR / "eval_predictions"
BASELINE_TEST_PREDICTIONS_PATH = PREDICTION_DIR / "mini_50h_100eval_baseline_test_predictions.jsonl"
BASELINE_TEST_METRICS_PATH = PREDICTION_DIR / "mini_50h_100eval_baseline_test_metrics.json"
FORCE_REGENERATE_BASELINE = False
MAX_BASELINE_TEST_SAMPLES = MINI_MAX_EVAL_SAMPLES
BASELINE_GENERATION_LANGUAGE = "Arabic"
SMOKE_CACHE_DIR = CACHE_DIR / "_smoke"

for p in [RUN_DIR, LOG_DIR, OUTPUT_DIR, BEST_MODEL_DIR, MANIFEST_DIR, CACHE_DIR, PREDICTION_DIR]:
    p.mkdir(parents=True, exist_ok=True)
for split in ["train", "val", "test"]:
    (CACHE_DIR / split).mkdir(parents=True, exist_ok=True)
    (MANIFEST_DIR / split).mkdir(parents=True, exist_ok=True)
    (SMOKE_CACHE_DIR / split).mkdir(parents=True, exist_ok=True)
(MANIFEST_DIR / "split_assignments").mkdir(parents=True, exist_ok=True)

CONFIG = {k: str(v) if isinstance(v, Path) else v for k, v in globals().items() if k.isupper() and k not in {"CONFIG"}}
print(f"Configured MINI run dir: {RUN_DIR}")
print(f"Notebook path: {NOTEBOOK_PATH}")
print(f"Train cap: {MINI_MAX_TRAIN_HOURS}h; eval/test/baseline cap: {MINI_MAX_EVAL_SAMPLES} samples")


## 2. Imports and Logging Setup


In [ ]:
from __future__ import annotations

import base64
import collections
import concurrent.futures
import contextlib
import dataclasses
import functools
import gc
import hashlib
import inspect
import io
import json
import logging
import math
import os
import platform
import queue
import random
import re
import shutil
import signal
import statistics
import subprocess
import sys
import tempfile
import threading
import time
import traceback
import unicodedata
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, Iterator, List, Optional, Sequence, Tuple

import numpy as np

try:
    import torch
    from torch.utils.data import Dataset, DataLoader, Sampler, BatchSampler
except Exception as exc:
    raise RuntimeError("PyTorch is required for this notebook.") from exc

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x=None, **kwargs):
        return x if x is not None else []

try:
    import soundfile as sf
except Exception:
    sf = None

try:
    import librosa
except Exception:
    librosa = None

try:
    import pyarrow as pa
    import pyarrow.ipc as pa_ipc
    import pyarrow.parquet as pq
except Exception:
    pa = pa_ipc = pq = None

try:
    from datasets import Dataset as HFDataset
except Exception:
    HFDataset = None

try:
    from transformers import GenerationConfig, Trainer, TrainerCallback, TrainingArguments
except Exception as exc:
    raise RuntimeError("transformers is required. For Qwen3-ASR, install/upgrade qwen-asr and transformers.") from exc

try:
    from peft import LoraConfig, TaskType, get_peft_model
except Exception as exc:
    raise RuntimeError("peft is required for LoRA fine-tuning: pip install -U peft") from exc

LOGGER_NAMES = ["main", "manifest", "cache", "train", "eval", "errors"]
LOGGERS = {}

def setup_logging() -> Dict[str, logging.Logger]:
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s")
    for name in LOGGER_NAMES:
        logger = logging.getLogger(name)
        logger.setLevel(logging.INFO)
        logger.handlers.clear()
        file_handler = logging.FileHandler(LOG_DIR / f"{name}.log", encoding="utf-8")
        file_handler.setFormatter(fmt)
        logger.addHandler(file_handler)
        if name == "main":
            stream = logging.StreamHandler(sys.stdout)
            stream.setFormatter(fmt)
            logger.addHandler(stream)
        logger.propagate = False
        LOGGERS[name] = logger
    LOGGERS["main"].info("Logging initialized at %s", LOG_DIR)
    LOGGERS["main"].info("Config: %s", json.dumps(CONFIG, ensure_ascii=False, indent=2, default=str))
    return LOGGERS

setup_logging()

def log_exception(where: str, exc: BaseException, row: Optional[dict] = None) -> None:
    payload = {"where": where, "error": repr(exc), "row_uid": row.get("uid") if row else None}
    LOGGERS["errors"].error("%s\n%s", json.dumps(payload, ensure_ascii=False), traceback.format_exc())
    if DEBUG_STRICT:
        raise exc

def jsonl_write(path: Path, rows: Iterable[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n")
        f.flush()
        os.fsync(f.fileno())
    tmp.replace(path)

def jsonl_read(path: Path) -> List[dict]:
    with Path(path).open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

print("Imports and logging are ready.")


## 3. Environment and GPU Checks


In [ ]:
def package_version(name: str) -> str:
    try:
        import importlib.metadata as md
        return md.version(name)
    except Exception:
        return "not-installed"

def check_environment() -> dict:
    info = {
        "python": sys.version.replace("\n", " "),
        "platform": platform.platform(),
        "cpu_count": os.cpu_count(),
        "torch": package_version("torch"),
        "transformers": package_version("transformers"),
        "qwen_asr": package_version("qwen-asr"),
        "peft": package_version("peft"),
        "datasets": package_version("datasets"),
        "pyarrow": package_version("pyarrow"),
        "librosa": package_version("librosa"),
        "soundfile": package_version("soundfile"),
        "cuda_available": torch.cuda.is_available(),
    }
    if torch.cuda.is_available():
        info.update({
            "cuda_device_count": torch.cuda.device_count(),
            "cuda_device_name": torch.cuda.get_device_name(0),
            "cuda_capability": torch.cuda.get_device_capability(0),
            "bf16_supported": bool(torch.cuda.is_bf16_supported()),
        })
    LOGGERS["main"].info("Environment: %s", json.dumps(info, ensure_ascii=False, indent=2, default=str))
    print(json.dumps(info, ensure_ascii=False, indent=2, default=str))
    return info

ENV_INFO = check_environment()
USE_BF16_EFFECTIVE = bool(USE_BF16 and torch.cuda.is_available() and torch.cuda.is_bf16_supported())
USE_FP16_EFFECTIVE = bool((USE_FP16 or not USE_BF16_EFFECTIVE) and torch.cuda.is_available())
MODEL_DTYPE = torch.bfloat16 if USE_BF16_EFFECTIVE else (torch.float16 if USE_FP16_EFFECTIVE else torch.float32)
print("Chosen model dtype:", MODEL_DTYPE)


## 4. Source Path Checks and Recursive File Discovery


In [ ]:
@dataclass(frozen=True)
class SourceRoot:
    path: Path
    source_type: str  # train or heldout
    source_group: str

def path_key(p: Path) -> str:
    return str(Path(p).resolve())

def source_group_for_root(root: Path, source_type: str) -> str:
    s = str(root)
    if "casablanca/relevant_arabic" in s:
        return "casablanca_relevant_arabic"
    if "omnilingual_selected/other_arabic_dialects" in s:
        return "omnilingual_other_arabic_dialects"
    if "apc_north_levantine_all_splits" in s:
        return "heldout_omnilingual_apc"
    if "casablanca/levant" in s:
        return "heldout_casablanca_levant"
    if "QASR" in s:
        return "qasr"
    return f"{source_type}_" + re.sub(r"[^a-zA-Z0-9]+", "_", root.name).strip("_").lower()

SOURCE_ROOTS = [SourceRoot(Path(p), "train", source_group_for_root(Path(p), "train")) for p in TRAIN_SOURCE_DIRS]
SOURCE_ROOTS += [SourceRoot(Path(p), "heldout", source_group_for_root(Path(p), "heldout")) for p in HELDOUT_VAL_TEST_SOURCE_DIRS]
if INCLUDE_QASR_IN_FULL_MANIFEST:
    SOURCE_ROOTS.append(SourceRoot(QASR_ROOT, "train", "qasr"))

def check_source_paths() -> None:
    for sr in SOURCE_ROOTS:
        exists = sr.path.exists()
        LOGGERS["main"].info("source root exists=%s type=%s group=%s path=%s", exists, sr.source_type, sr.source_group, sr.path)
        print(f"{sr.source_type:7s} {sr.source_group:35s} exists={exists} {sr.path}")

AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg", ".m4a", ".opus"}

def discover_files(root: Path) -> Dict[str, List[Path]]:
    found = {"arrow": [], "parquet": [], "xml": [], "wav": [], "audio": []}
    if not root.exists():
        return found
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        suffix = p.suffix.lower()
        if suffix == ".arrow":
            found["arrow"].append(p)
        elif suffix == ".parquet":
            found["parquet"].append(p)
        elif suffix == ".xml":
            found["xml"].append(p)
        elif suffix == ".wav":
            found["wav"].append(p)
            found["audio"].append(p)
        elif suffix in AUDIO_EXTS:
            found["audio"].append(p)
    for k in found:
        found[k].sort()
    return found

check_source_paths()
DISCOVERED = {path_key(sr.path): discover_files(sr.path) for sr in SOURCE_ROOTS}
for sr in SOURCE_ROOTS:
    counts = {k: len(v) for k, v in DISCOVERED[path_key(sr.path)].items()}
    LOGGERS["main"].info("discovered root=%s counts=%s", sr.path, counts)
    print(sr.source_group, counts)


## 5. Arabic Normalization

Aggressive ASR normalization is off by default. The same function is used for labels and metric text, with knobs in the config cell.


In [ ]:
AR_DIACRITICS_RE = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
CONTROL_RE = re.compile(r"[\u0000-\u001F\u007F-\u009F\u200e\u200f\u202a-\u202e]")
SPACE_RE = re.compile(r"\s+")
ASR_TAG_RE = re.compile(r"<[^>]+>|\[[^\]]+\]")
AR_PUNCT_SPACING_RE = re.compile(r"\s*([،؛؟,.!?;:])\s*")
REPEATED_PUNCT_RE = re.compile(r"([،؛؟,.!?;:])\1+")
PUNCT_RE = re.compile(r"[\.,!\?;:،؛؟\-\(\)\[\]{}\"'“”‘’«»]")
WESTERN_TO_ARABIC_PUNCT = str.maketrans({",": "،", ";": "؛", "?": "؟"})


def normalize_arabic_text(text: Any, *, for_metrics: bool = False, remove_punctuation: Optional[bool] = None) -> str:
    """Normalize dataset artifacts while preserving Arabic spelling by default.

    Training labels remove ASR/noise tags, tatweel, control/bidi marks, harakat, and spacing noise.
    They keep hamza forms, alef maqsura, taa marbuta, and punctuation unless explicitly configured.
    """
    if text is None:
        return ""
    before = str(text)
    text = unicodedata.normalize("NFKC", before)
    text = CONTROL_RE.sub(" ", text)
    if REMOVE_ASR_TAGS:
        text = ASR_TAG_RE.sub(" ", text)
    if REMOVE_TATWEEL:
        text = text.replace("ـ", "")
    if REMOVE_DIACRITICS:
        text = AR_DIACRITICS_RE.sub("", text)
    if NORMALIZE_ALEF:
        text = re.sub("[إأآٱ]", "ا", text)
    if NORMALIZE_YAA:
        text = text.replace("ى", "ي")
    if NORMALIZE_TAA_MARBUTA:
        text = text.replace("ة", "ه")
    if MAP_WESTERN_TO_ARABIC_PUNCT:
        text = text.translate(WESTERN_TO_ARABIC_PUNCT)
    if NORMALIZE_ARABIC:
        text = AR_PUNCT_SPACING_RE.sub(r" \1 ", text)
    if COLLAPSE_REPEATED_PUNCT:
        text = REPEATED_PUNCT_RE.sub(r"\1", text)
    do_remove_punctuation = REMOVE_PUNCTUATION if remove_punctuation is None else remove_punctuation
    if do_remove_punctuation:
        text = PUNCT_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text).strip()
    if before != text:
        LOGGERS["main"].debug("normalize_arabic before=%r after=%r", before, text)
    return text


def normalize_metric_text(text: Any, *, loose: bool = False, punctuation_insensitive: bool = False) -> str:
    text = normalize_arabic_text(text, for_metrics=True, remove_punctuation=punctuation_insensitive)
    if loose:
        text = re.sub("[إأآٱ]", "ا", text)
        text = text.replace("ى", "ي")
        text = SPACE_RE.sub(" ", text).strip()
    return text

_examples = ["  مرحباً،هذا   اختبار\u200f ", "السلامُ عليكم ؟", "كيف الحال!!!", "<hesitation> أهلاً ـــ وسهلاً [music]"]
for ex in _examples:
    print(repr(ex), "=>", repr(normalize_arabic_text(ex)))


## 6. Dataset-Specific Parsers and Adapters

Each parser emits rows in the unified lightweight manifest format. Large audio payloads are never embedded in the manifest.


In [ ]:
SPLIT_ALIASES = {
    "validation": "val", "valid": "val", "val": "val", "eval": "val", "dev": "val",
    "test": "test", "testing": "test",
    "train": "train", "training": "train",
}

def infer_original_split(path: Path, metadata_split: Any = None) -> str:
    candidates = []
    if metadata_split:
        candidates.append(str(metadata_split).lower())
    candidates.extend(part.lower() for part in Path(path).parts)
    candidates.append(Path(path).name.lower())
    for c in candidates:
        for key in ["validation", "valid", "val", "eval", "dev", "test", "train", "training"]:
            if re.search(rf"(^|[^a-z]){re.escape(key)}([^a-z]|$)", c):
                return key
    return "unknown"

def stable_hash(text: str, seed: int = SPLIT_SEED) -> str:
    h = hashlib.sha1()
    h.update(str(seed).encode("utf-8"))
    h.update(b"|")
    h.update(text.encode("utf-8"))
    return h.hexdigest()

def stable_uid(*parts: Any) -> str:
    return stable_hash("|".join(str(p) for p in parts), seed=0)[:24]

def detect_source_kind(path: Path) -> str:
    s = str(path).lower()
    if path.suffix.lower() == ".arrow" or "omnilingual" in s:
        return "omnilingual"
    if path.suffix.lower() == ".parquet" or "casablanca" in s:
        return "casablanca"
    if path.suffix.lower() == ".xml" or "qasr" in s:
        return "qasr"
    return "unknown"

def base_manifest_row(**kwargs) -> dict:
    normalized_text = normalize_arabic_text(kwargs.pop("text", ""))
    row = {
        "uid": kwargs.pop("uid"),
        "source": kwargs.pop("source"),
        "source_root": str(kwargs.pop("source_root")),
        "source_group": kwargs.pop("source_group"),
        "original_split": kwargs.pop("original_split", "unknown"),
        "split": kwargs.pop("split", None),
        "text": normalized_text,
        "duration": kwargs.pop("duration", None),
        "audio_kind": kwargs.pop("audio_kind"),
        "audio_path": kwargs.pop("audio_path", None),
        "audio_bytes_ref": kwargs.pop("audio_bytes_ref", None),
        "arrow_file": kwargs.pop("arrow_file", None),
        "parquet_file": kwargs.pop("parquet_file", None),
        "row_idx": kwargs.pop("row_idx", None),
        "start": kwargs.pop("start", None),
        "end": kwargs.pop("end", None),
        "language": kwargs.pop("language", None),
        "speaker_id": kwargs.pop("speaker_id", None),
        "gender": kwargs.pop("gender", None),
        "segment_id": kwargs.pop("segment_id", None),
        "metadata": kwargs.pop("metadata", {}),
    }
    row.update(kwargs)
    return row

def valid_duration(duration: Any) -> bool:
    if duration is None:
        return True
    try:
        d = float(duration)
    except Exception:
        return False
    if d <= 0 or d < MIN_AUDIO_SECONDS:
        return False
    if MAX_AUDIO_SECONDS is not None and d > MAX_AUDIO_SECONDS:
        return False
    return True

def iter_arrow_pyrows(path: Path, columns: Optional[List[str]] = None):
    if pa is None or pa_ipc is None:
        raise RuntimeError("pyarrow is required to read HF Arrow files without decoding audio")
    with pa.memory_map(str(path), "r") as source:
        reader = pa_ipc.open_stream(source)
        row_base = 0
        for batch in reader:
            names = set(batch.schema.names)
            use_cols = [c for c in (columns or batch.schema.names) if c in names]
            data = {c: batch.column(batch.schema.get_field_index(c)).to_pylist() for c in use_cols}
            for j in range(batch.num_rows):
                yield row_base + j, {c: data[c][j] for c in use_cols}
            row_base += batch.num_rows

def parse_omnilingual_arrow_file(path: Path, root: SourceRoot, limit: Optional[int] = None) -> List[dict]:
    rows = []
    try:
        columns = ["language", "speaker_id", "prompt_id", "prompt", "segment_id", "duration", "raw_text", "iso_639_3", "glottocode", "iso_15924", "config", "original_split"]
        logged_schema = False
        for row_idx, ex in tqdm(iter_arrow_pyrows(path, columns), desc=f"arrow {path.name}"):
            if not logged_schema:
                LOGGERS["manifest"].info("Arrow metadata reader opened %s", path)
                logged_schema = True
            if limit is not None and len(rows) >= limit:
                break
            text = ex.get("raw_text") or ex.get("text") or ex.get("transcript") or ""
            if not text or not valid_duration(ex.get("duration")):
                continue
            normalized = normalize_arabic_text(text)
            if DROP_EMPTY_AFTER_NORMALIZATION and not normalized:
                continue
            original_split = infer_original_split(path, ex.get("original_split"))
            row = base_manifest_row(
                uid=stable_uid("arrow", path.resolve(), row_idx, ex.get("segment_id", "")),
                source="omnilingual",
                source_root=root.path,
                source_group=root.source_group,
                original_split=original_split,
                text=normalized,
                duration=ex.get("duration"),
                audio_kind="hf_audio",
                arrow_file=str(path),
                row_idx=row_idx,
                language=ex.get("language") or ex.get("iso_639_3"),
                speaker_id=ex.get("speaker_id"),
                segment_id=ex.get("segment_id"),
                metadata={k: ex.get(k) for k in ["prompt_id", "prompt", "iso_639_3", "glottocode", "iso_15924", "config"] if k in ex},
            )
            rows.append(row)
    except Exception as exc:
        log_exception(f"parse_omnilingual_arrow_file:{path}", exc)
    return rows

def parquet_audio_kind(audio_obj: Any) -> Tuple[str, Optional[str]]:
    if isinstance(audio_obj, dict):
        if audio_obj.get("bytes") is not None:
            return "bytes", None
        if audio_obj.get("path"):
            return "path", str(audio_obj.get("path"))
    return "bytes", None

def parse_casablanca_parquet_file(path: Path, root: SourceRoot, limit: Optional[int] = None) -> List[dict]:
    if pq is None:
        raise RuntimeError("pyarrow is required to read Casablanca Parquet")
    rows = []
    try:
        pf = pq.ParquetFile(path)
        LOGGERS["manifest"].info("Parquet schema %s: %s", path, pf.schema)
        seen = 0
        for batch in pf.iter_batches(batch_size=1024):
            batch_rows = batch.to_pylist()
            for ex in batch_rows:
                if limit is not None and seen >= limit:
                    return rows
                row_idx = seen
                seen += 1
                text = ex.get("transcription") or ex.get("text") or ""
                duration = ex.get("duration")
                if not text or not valid_duration(duration):
                    continue
                normalized = normalize_arabic_text(text)
                if DROP_EMPTY_AFTER_NORMALIZATION and not normalized:
                    continue
                audio_obj = ex.get("audio") or {}
                audio_kind, audio_path = parquet_audio_kind(audio_obj)
                row = base_manifest_row(
                    uid=stable_uid("parquet", path.resolve(), row_idx, ex.get("seg_id", "")),
                    source="casablanca",
                    source_root=root.path,
                    source_group=root.source_group,
                    original_split=infer_original_split(path),
                    text=normalized,
                    duration=duration,
                    audio_kind=audio_kind,
                    audio_path=audio_path,
                    audio_bytes_ref={"parquet_file": str(path), "row_idx": row_idx, "column": "audio.bytes"} if audio_kind == "bytes" else None,
                    parquet_file=str(path),
                    row_idx=row_idx,
                    gender=ex.get("gender"),
                    segment_id=ex.get("seg_id"),
                    metadata={k: v for k, v in ex.items() if k not in {"audio", "transcription"}},
                )
                rows.append(row)
    except Exception as exc:
        log_exception(f"parse_casablanca_parquet_file:{path}", exc)
    return rows

def strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag

def find_qasr_wav(xml_path: Path, wav_index: Dict[str, Path]) -> Optional[Path]:
    stem = xml_path.stem
    return wav_index.get(stem.lower())

def parse_qasr_xml_file(xml_path: Path, root: SourceRoot, wav_index: Dict[str, Path], limit: Optional[int] = None) -> List[dict]:
    rows = []
    wav_path = find_qasr_wav(xml_path, wav_index)
    if wav_path is None:
        LOGGERS["manifest"].warning("QASR missing wav for %s", xml_path)
        return rows
    try:
        tree = ET.parse(xml_path)
        root_el = tree.getroot()
        speaker_meta = {}
        for spk in root_el.iter():
            if strip_ns(spk.tag) == "speaker" and spk.attrib.get("id"):
                speaker_meta[spk.attrib["id"]] = dict(spk.attrib)
        count = 0
        for segs in root_el.iter():
            if strip_ns(segs.tag) != "segments" or segs.attrib.get("annotation_id") != "transcript_align":
                continue
            for seg in segs:
                if strip_ns(seg.tag) != "segment":
                    continue
                try:
                    start = float(seg.attrib.get("starttime", "nan"))
                    end = float(seg.attrib.get("endtime", "nan"))
                except Exception:
                    continue
                duration = end - start
                if not valid_duration(duration):
                    continue
                tokens = []
                for child in seg:
                    child_type = child.attrib.get("type", "")
                    if child_type in {"word", "punctuation", "punct"} or child.text:
                        txt = (child.text or "").strip()
                        if txt:
                            tokens.append(txt)
                text = normalize_arabic_text(" ".join(tokens))
                if not text:
                    continue
                speaker_id = seg.attrib.get("who")
                rows.append(base_manifest_row(
                    uid=stable_uid("qasr", xml_path.resolve(), seg.attrib.get("id", ""), start, end),
                    source="qasr",
                    source_root=root.path,
                    source_group=root.source_group,
                    original_split=infer_original_split(xml_path),
                    text=text,
                    duration=duration,
                    audio_kind="path",
                    audio_path=str(wav_path),
                    start=start,
                    end=end,
                    speaker_id=speaker_id,
                    segment_id=seg.attrib.get("id"),
                    metadata={"xml_file": str(xml_path), "speaker": speaker_meta.get(speaker_id, {})},
                ))
                count += 1
                if limit is not None and count >= limit:
                    return rows
    except Exception as exc:
        log_exception(f"parse_qasr_xml_file:{xml_path}", exc)
    return rows

print("Parser/adapters are defined.")


## 7. Deterministic Split Assignment

Saved split assignments are reused exactly unless `FORCE_RESPLIT=True`.


In [ ]:
VAL_UIDS_PATH = MANIFEST_DIR / "split_assignments" / "val_uids.txt"
TEST_UIDS_PATH = MANIFEST_DIR / "split_assignments" / "test_uids.txt"
ASSIGNMENTS_JSONL_PATH = MANIFEST_DIR / "split_assignments" / "split_assignments.jsonl"

def read_uid_file(path: Path) -> set:
    if not path.exists():
        return set()
    return {line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()}

def save_split_assignments(rows: List[dict]) -> None:
    val_uids = sorted(r["uid"] for r in rows if r.get("split") == "val")
    test_uids = sorted(r["uid"] for r in rows if r.get("split") == "test")
    VAL_UIDS_PATH.write_text("\n".join(val_uids) + ("\n" if val_uids else ""), encoding="utf-8")
    TEST_UIDS_PATH.write_text("\n".join(test_uids) + ("\n" if test_uids else ""), encoding="utf-8")
    jsonl_write(ASSIGNMENTS_JSONL_PATH, [
        {"uid": r["uid"], "split": r.get("split"), "source_root": r.get("source_root"), "source_group": r.get("source_group"), "original_split": r.get("original_split")}
        for r in rows if r.get("split") in {"val", "test"}
    ])

def load_saved_assignments() -> Dict[str, str]:
    assignments = {}
    for uid in read_uid_file(VAL_UIDS_PATH):
        assignments[uid] = "val"
    for uid in read_uid_file(TEST_UIDS_PATH):
        assignments[uid] = "test"
    if ASSIGNMENTS_JSONL_PATH.exists():
        for row in jsonl_read(ASSIGNMENTS_JSONL_PATH):
            assignments[row["uid"]] = row["split"]
    return assignments

def assign_splits(rows: List[dict]) -> List[dict]:
    saved = load_saved_assignments() if (not FORCE_RESPLIT and VAL_UIDS_PATH.exists() and TEST_UIDS_PATH.exists()) else {}
    out = []
    for row in rows:
        row = dict(row)
        source_root = Path(row["source_root"])
        root_type = "train"
        for sr in SOURCE_ROOTS:
            try:
                if source_root.resolve() == sr.path.resolve():
                    root_type = sr.source_type
                    break
            except Exception:
                pass
        original = (row.get("original_split") or "unknown").lower()
        mapped = SPLIT_ALIASES.get(original, "unknown")
        if root_type == "train":
            row["split"] = "train"
        else:
            if row["uid"] in saved:
                row["split"] = saved[row["uid"]]
            elif mapped in {"val", "test"}:
                row["split"] = mapped
            else:
                frac = int(stable_hash(row["uid"], SPLIT_SEED)[:12], 16) / float(16**12)
                row["split"] = "val" if frac < VAL_FRACTION_OF_UNSPLIT_HELDOUT else "test"
        out.append(row)
    save_split_assignments(out)
    return out

def log_manifest_counts(rows: List[dict], title: str = "manifest") -> None:
    keys = ["source_root", "source", "source_group", "original_split", "split"]
    print(f"\n{title}: {len(rows)} rows")
    LOGGERS["manifest"].info("%s total=%d", title, len(rows))
    for key in keys:
        counter = collections.Counter(str(r.get(key)) for r in rows)
        print(f"-- {key}")
        for k, v in counter.most_common():
            print(f"   {k}: {v}")
        LOGGERS["manifest"].info("%s by %s: %s", title, key, dict(counter))
    durations = [float(r["duration"]) for r in rows if r.get("duration") is not None]
    if durations:
        stats = {"min": min(durations), "max": max(durations), "mean": statistics.mean(durations), "sum_hours": sum(durations) / 3600.0}
        print("duration stats:", stats)
        LOGGERS["manifest"].info("%s duration_stats=%s", title, stats)

print("Split assignment helpers are ready.")


## 8. Unified Manifest Builder


In [ ]:
MANIFEST_ALL_PATH = MANIFEST_DIR / "manifest_all.jsonl"
MANIFEST_SPLIT_PATHS = {
    "train": MANIFEST_DIR / "train" / "manifest_train.jsonl",
    "val": MANIFEST_DIR / "val" / "manifest_val.jsonl",
    "test": MANIFEST_DIR / "test" / "manifest_test.jsonl",
}
MANIFEST_SMOKE_PATH = MANIFEST_DIR / "manifest_smoke.jsonl"

def build_manifest_rows_for_root(sr: SourceRoot, limits: Optional[dict] = None) -> List[dict]:
    limits = limits or {}
    files = discover_files(sr.path)
    rows = []
    for arrow_file in tqdm(files["arrow"], desc=f"scan arrow {sr.source_group}"):
        rows.extend(parse_omnilingual_arrow_file(arrow_file, sr, limit=limits.get("arrow")))
        if limits.get("arrow_total") and len([r for r in rows if r["source"] == "omnilingual"]) >= limits["arrow_total"]:
            break
    for parquet_file in tqdm(files["parquet"], desc=f"scan parquet {sr.source_group}"):
        rows.extend(parse_casablanca_parquet_file(parquet_file, sr, limit=limits.get("parquet")))
        if limits.get("parquet_total") and len([r for r in rows if r["source"] == "casablanca"]) >= limits["parquet_total"]:
            break
    if sr.source_group == "qasr" or files["xml"]:
        wav_index = {p.stem.lower(): p for p in files["wav"]}
        qasr_count = 0
        for xml_file in tqdm(files["xml"], desc=f"scan qasr {sr.source_group}"):
            got = parse_qasr_xml_file(xml_file, sr, wav_index, limit=limits.get("qasr_segments"))
            rows.extend(got)
            qasr_count += len(got)
            if limits.get("qasr_total") and qasr_count >= limits["qasr_total"]:
                break
    return rows

def write_manifests(rows: List[dict]) -> None:
    jsonl_write(MANIFEST_ALL_PATH, rows)
    for split, path in MANIFEST_SPLIT_PATHS.items():
        jsonl_write(path, [r for r in rows if r.get("split") == split])
    log_manifest_counts(rows, "full manifest")

def build_or_load_full_manifest() -> List[dict]:
    if MANIFEST_ALL_PATH.exists() and not FORCE_REBUILD_MANIFEST:
        rows = jsonl_read(MANIFEST_ALL_PATH)
        print(f"Loaded existing manifest: {MANIFEST_ALL_PATH} rows={len(rows)}")
        log_manifest_counts(rows, "loaded manifest")
        return rows
    all_rows = []
    for sr in SOURCE_ROOTS:
        all_rows.extend(build_manifest_rows_for_root(sr))
    all_rows = assign_splits(all_rows)
    write_manifests(all_rows)
    return all_rows

print("Manifest builder is ready.")


## 9. Audio Loading Utilities

The lazy dataset can resolve HF Arrow Audio, Parquet bytes, direct paths, and QASR crops. Per-worker handle caches avoid reopening heavy files for every row.


In [ ]:
_WORKER_STATE = threading.local()

def worker_state() -> dict:
    if not hasattr(_WORKER_STATE, "state"):
        _WORKER_STATE.state = {"arrow": {}, "parquet": {}, "cache_shards": {}}
    return _WORKER_STATE.state

def ensure_audio_tools() -> None:
    if sf is None and librosa is None:
        raise RuntimeError("Install soundfile or librosa for audio decoding")

def to_mono_float32(audio: np.ndarray) -> np.ndarray:
    audio = np.asarray(audio)
    if audio.ndim == 2:
        audio = audio.mean(axis=1 if audio.shape[0] > audio.shape[1] else 0)
    audio = audio.astype(np.float32, copy=False)
    if audio.size and np.max(np.abs(audio)) > 1.5:
        audio = audio / np.iinfo(np.int16).max
    return np.ascontiguousarray(audio)

def resample_audio(audio: np.ndarray, sr: int, target_sr: int = SAMPLE_RATE) -> np.ndarray:
    audio = to_mono_float32(audio)
    if int(sr) == int(target_sr):
        return audio
    if librosa is None:
        raise RuntimeError("librosa is required for resampling")
    return librosa.resample(audio, orig_sr=int(sr), target_sr=int(target_sr)).astype(np.float32)

def crop_audio(audio: np.ndarray, start: Optional[float], end: Optional[float], sr: int = SAMPLE_RATE) -> np.ndarray:
    if start is None or end is None:
        return audio
    lo = max(0, int(float(start) * sr))
    hi = min(len(audio), int(float(end) * sr))
    return audio[lo:hi]

def read_audio_bytes(data: bytes) -> Tuple[np.ndarray, int]:
    ensure_audio_tools()
    if sf is not None:
        with io.BytesIO(data) as bio:
            audio, sr = sf.read(bio, dtype="float32", always_2d=False)
            return to_mono_float32(audio), int(sr)
    with tempfile.NamedTemporaryFile(suffix=".audio") as tmp:
        tmp.write(data); tmp.flush()
        audio, sr = librosa.load(tmp.name, sr=None, mono=True)
        return to_mono_float32(audio), int(sr)

def read_audio_path(path: str) -> Tuple[np.ndarray, int]:
    ensure_audio_tools()
    if sf is not None:
        audio, sr = sf.read(path, dtype="float32", always_2d=False)
        return to_mono_float32(audio), int(sr)
    audio, sr = librosa.load(path, sr=None, mono=True)
    return to_mono_float32(audio), int(sr)

def read_arrow_row_py(path: str, row_idx: int, columns: Optional[List[str]] = None) -> dict:
    if pa is None or pa_ipc is None:
        raise RuntimeError("pyarrow is required for Arrow audio rows")
    idx = int(row_idx)
    seen = 0
    with pa.memory_map(str(path), "r") as source:
        reader = pa_ipc.open_stream(source)
        for batch in reader:
            if seen + batch.num_rows <= idx:
                seen += batch.num_rows
                continue
            local = idx - seen
            names = set(batch.schema.names)
            use_cols = [c for c in (columns or batch.schema.names) if c in names]
            return {c: batch.column(batch.schema.get_field_index(c)).to_pylist()[local] for c in use_cols}
    raise IndexError(f"row_idx={row_idx} out of range for {path}")

def load_arrow_audio(row: dict) -> Tuple[np.ndarray, int]:
    ex = read_arrow_row_py(row["arrow_file"], int(row["row_idx"]), columns=["audio"])
    audio = ex.get("audio")
    if isinstance(audio, dict):
        if audio.get("array") is not None:
            sr = audio.get("sampling_rate") or SAMPLE_RATE
            return to_mono_float32(audio["array"]), int(sr)
        if audio.get("bytes") is not None:
            return read_audio_bytes(audio["bytes"])
        if audio.get("path"):
            p = audio["path"]
            if not Path(p).is_absolute():
                p = str(Path(row["arrow_file"]).parent / p)
            return read_audio_path(p)
    raise RuntimeError(f"Unsupported Arrow audio payload for uid={row.get('uid')}")

def read_parquet_row(path: str, row_idx: int) -> dict:
    if pq is None:
        raise RuntimeError("pyarrow is required for parquet audio rows")
    pf = pq.ParquetFile(path)
    seen = 0
    for batch in pf.iter_batches(batch_size=1024):
        b = batch.to_pylist()
        if seen + len(b) > row_idx:
            return b[row_idx - seen]
        seen += len(b)
    raise IndexError(f"row_idx={row_idx} out of range for {path}")

def load_parquet_audio(row: dict) -> Tuple[np.ndarray, int]:
    ex = read_parquet_row(row["parquet_file"], int(row["row_idx"]))
    audio = ex.get("audio") or {}
    if isinstance(audio, dict):
        if audio.get("bytes") is not None:
            return read_audio_bytes(audio["bytes"])
        if audio.get("path"):
            p = audio["path"]
            if not Path(p).is_absolute():
                p = str(Path(row["parquet_file"]).parent / p)
            return read_audio_path(p)
    raise RuntimeError(f"Unsupported Parquet audio payload for uid={row.get('uid')}")

def load_audio_for_row(row: dict) -> Tuple[np.ndarray, int]:
    if row.get("audio_kind") == "array":
        return to_mono_float32(np.asarray(row["audio_array"], dtype=np.float32)), int(row.get("sampling_rate", SAMPLE_RATE))
    if row.get("audio_kind") == "hf_audio":
        audio, sr = load_arrow_audio(row)
    elif row.get("audio_kind") == "bytes":
        audio, sr = load_parquet_audio(row)
    elif row.get("audio_kind") == "path" and row.get("audio_path"):
        audio, sr = read_audio_path(row["audio_path"])
    else:
        raise RuntimeError(f"Unsupported audio_kind={row.get('audio_kind')} uid={row.get('uid')}")
    audio = resample_audio(audio, sr, SAMPLE_RATE)
    audio = crop_audio(audio, row.get("start"), row.get("end"), SAMPLE_RATE)
    return audio, SAMPLE_RATE

print("Audio utilities are ready.")


## 10. Qwen3-ASR Processor and Model Loading

This follows the official Qwen3-ASR fine-tuning path: load `qwen_asr.Qwen3ASRModel`, extract `.model` and `.processor`, then patch the outer model forward to delegate to `model.thinker.forward`.


In [ ]:
def patch_outer_forward(model):
    cls = model.__class__
    if getattr(cls, "_forward_patched", False):
        return
    if not hasattr(model, "thinker") or not hasattr(model.thinker, "forward"):
        raise RuntimeError("Cannot patch forward: Qwen3-ASR model has no `.thinker.forward`.")
    def forward(self, input_ids=None, attention_mask=None, input_features=None, feature_attention_mask=None, labels=None, **kwargs):
        return self.thinker.forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            input_features=input_features,
            feature_attention_mask=feature_attention_mask,
            labels=labels,
            **kwargs,
        )
    cls.forward = forward
    cls._forward_patched = True

def load_qwen3_asr_model_and_processor():
    from qwen_asr import Qwen3ASRModel
    LOGGERS["main"].info("Loading Qwen3-ASR model %s dtype=%s", MODEL_NAME, MODEL_DTYPE)
    asr_wrapper = Qwen3ASRModel.from_pretrained(
        MODEL_NAME,
        dtype=MODEL_DTYPE,
        device_map=None,
        max_new_tokens=256,
        max_inference_batch_size=8,
    )
    model = asr_wrapper.model
    processor = asr_wrapper.processor
    patch_outer_forward(model)
    model.generation_config = GenerationConfig.from_model_config(model.config)
    if torch.cuda.is_available():
        model.to("cuda")
    print("model class:", model.__class__)
    print("processor class:", processor.__class__)
    print("tokenizer class:", getattr(processor, "tokenizer", None).__class__ if hasattr(processor, "tokenizer") else None)
    print("feature extractor class:", getattr(processor, "feature_extractor", None).__class__ if hasattr(processor, "feature_extractor") else None)
    print("model config model_type:", getattr(model.config, "model_type", None))
    print("forward signature:", inspect.signature(model.forward))
    LOGGERS["main"].info("model=%s processor=%s tokenizer=%s feature_extractor=%s", model.__class__, processor.__class__, getattr(processor, "tokenizer", None).__class__ if hasattr(processor, "tokenizer") else None, getattr(processor, "feature_extractor", None).__class__ if hasattr(processor, "feature_extractor") else None)
    return asr_wrapper, model, processor

asr_wrapper, model, processor = load_qwen3_asr_model_and_processor()


## 11. LoRA Setup

Target modules are discovered from actual Qwen3-ASR module names. The preferred projection names are `q_proj`, `k_proj`, `v_proj`, and `o_proj` when present.


In [ ]:
def infer_lora_target_modules(model) -> List[str]:
    if LORA_TARGET_MODULES != "auto":
        return list(LORA_TARGET_MODULES)
    preferred = ["q_proj", "k_proj", "v_proj", "o_proj"]
    present = set()
    linear_suffixes = collections.Counter()
    for name, module in model.named_modules():
        leaf = name.split(".")[-1]
        if isinstance(module, torch.nn.Linear):
            linear_suffixes[leaf] += 1
            if leaf in preferred:
                present.add(leaf)
    if present:
        targets = [x for x in preferred if x in present]
    else:
        fallback = [name for name, count in linear_suffixes.most_common() if any(tok in name for tok in ["proj", "linear", "fc"])]
        targets = fallback[:8]
    if not targets:
        raise RuntimeError("Could not infer LoRA target modules. Inspect model.named_modules().")
    print("LoRA target modules:", targets)
    LOGGERS["train"].info("LoRA targets: %s", targets)
    return targets

def count_parameters(model) -> dict:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "pct_trainable": 100.0 * trainable / max(1, total)}

def apply_lora(model):
    targets = infer_lora_target_modules(model)
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=targets,
        task_type=TaskType.CAUSAL_LM,
    )
    peft_model = get_peft_model(model, lora_config)
    peft_model.print_trainable_parameters()
    stats = count_parameters(peft_model)
    print("parameter stats:", stats)
    LOGGERS["train"].info("parameter_stats=%s", stats)
    return peft_model, lora_config

model, lora_config = apply_lora(model)


## 12. Qwen3-ASR Sample Processing and Collator

The label construction mirrors the official SFT script: build an audio prompt with `processor.apply_chat_template`, append `language Arabic<asr_text>...`, and mask the prompt prefix with `-100`.


In [ ]:
def build_prefix_messages(prompt: str, audio_array):
    return [
        {"role": "system", "content": prompt or ""},
        {"role": "user", "content": [{"type": "audio", "audio": audio_array}]},
    ]

def qwen_language_prefix(row: dict) -> str:
    lang = row.get("language") or "Arabic"
    if str(lang).lower() in {"ar", "ara", "arabic"}:
        lang = "Arabic"
    return f"language {lang}<asr_text>"

def build_prefix_text(processor, row: dict) -> str:
    prompt = row.get("prompt", "") or row.get("metadata", {}).get("prompt", "") or ""
    messages = build_prefix_messages(prompt, None)
    try:
        templated = processor.apply_chat_template([messages], add_generation_prompt=True, tokenize=False)
        return templated[0] if isinstance(templated, list) else templated
    except TypeError:
        return processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

def squeeze_batch_dim(batch: Dict[str, Any]) -> Dict[str, Any]:
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v) and v.ndim > 0 and v.shape[0] == 1:
            out[k] = v[0].detach().cpu()
        else:
            out[k] = v.detach().cpu() if torch.is_tensor(v) else v
    return out

def process_manifest_row(row: dict, processor) -> dict:
    audio, sr = load_audio_for_row(row)
    text = normalize_arabic_text(row.get("text", ""))
    target = qwen_language_prefix(row) + text
    eos = getattr(processor.tokenizer, "eos_token", "") or ""
    prefix_text = build_prefix_text(processor, row)
    full_text = prefix_text + target + eos
    full_inputs = processor(text=[full_text], audio=[audio], return_tensors="pt", padding=True, truncation=False)
    prefix_inputs = processor(text=[prefix_text], audio=[audio], return_tensors="pt", padding=True, truncation=False)
    labels = full_inputs["input_ids"].clone()
    prefix_len = int(prefix_inputs["attention_mask"].sum(dim=1).item())
    labels[0, :prefix_len] = -100
    pad_id = getattr(processor.tokenizer, "pad_token_id", None)
    if pad_id is not None:
        labels[labels == pad_id] = -100
    full_inputs["labels"] = labels
    sample = squeeze_batch_dim(full_inputs)
    sample.update({
        "uid": row["uid"],
        "text": text,
        "source": row.get("source"),
        "source_root": row.get("source_root"),
        "source_group": row.get("source_group"),
        "split": row.get("split"),
        "duration": row.get("duration"),
        "metadata": row.get("metadata", {}),
    })
    return sample

def pad_tensor_list(values: List[torch.Tensor], pad_value: int | float = 0):
    values = [v if torch.is_tensor(v) else torch.tensor(v) for v in values]
    if not values:
        return torch.tensor([])
    max_ndim = max(v.ndim for v in values)
    values = [v.reshape((1,) if v.ndim == 0 else v.shape) for v in values]
    if max_ndim <= 1:
        max_len = max(v.shape[0] for v in values)
        out = values[0].new_full((len(values), max_len), pad_value)
        for i, v in enumerate(values):
            out[i, :v.shape[0]] = v
        return out
    if max_ndim == 2:
        d0 = max(v.shape[0] for v in values)
        d1 = max(v.shape[1] for v in values)
        out = values[0].new_full((len(values), d0, d1), pad_value)
        for i, v in enumerate(values):
            out[i, :v.shape[0], :v.shape[1]] = v
        return out
    raise ValueError(f"Cannot pad tensors with ndim={max_ndim}")

@dataclass
class DataCollatorForQwen3ASRLoRA:
    processor: Any

    def __call__(self, features: List[dict]) -> Dict[str, torch.Tensor]:
        keys = ["input_ids", "attention_mask", "input_features", "feature_attention_mask", "labels"]
        batch = {}
        pad_id = getattr(self.processor.tokenizer, "pad_token_id", 0) or 0
        for key in keys:
            vals = [f[key] for f in features if key in f and f[key] is not None]
            if not vals:
                continue
            if key == "labels":
                batch[key] = pad_tensor_list(vals, -100).long()
            elif key == "input_ids":
                batch[key] = pad_tensor_list(vals, pad_id).long()
            elif key.endswith("attention_mask"):
                batch[key] = pad_tensor_list(vals, 0).long()
            else:
                batch[key] = pad_tensor_list(vals, 0.0).float()
        batch["uids"] = [f.get("uid") for f in features]
        batch["source_groups"] = [f.get("source_group") for f in features]
        return batch

collator = DataCollatorForQwen3ASRLoRA(processor)
print("Collator is ready.")


## 13. Cache Shard and Index Utilities

Completed cache shards are immutable unless `FORCE_REBUILD_CACHE=True`. `.tmp` shards are ignored and can be cleaned safely.


In [ ]:
CACHE_STATS = collections.Counter()

def split_cache_dir(split: str, base: Path = CACHE_DIR) -> Path:
    d = Path(base) / split
    d.mkdir(parents=True, exist_ok=True)
    return d

def cache_index_path(split: str, base: Path = CACHE_DIR) -> Path:
    return split_cache_dir(split, base) / "cache_index.jsonl"

def tensor_to_obj(v: Any) -> Any:
    if torch.is_tensor(v):
        return v.detach().cpu().tolist()
    if isinstance(v, np.ndarray):
        return v.tolist()
    return v

def obj_to_tensor(k: str, v: Any) -> Any:
    if k in {"input_ids", "attention_mask", "labels", "feature_attention_mask"}:
        return torch.tensor(v, dtype=torch.long)
    if k in {"input_features"}:
        return torch.tensor(v, dtype=torch.float32)
    return v

def sample_to_cache_row(sample: dict) -> dict:
    row = {}
    for k, v in sample.items():
        row[k] = tensor_to_obj(v)
    return row

def cache_row_to_sample(row: dict) -> dict:
    return {k: obj_to_tensor(k, v) for k, v in row.items()}

def write_cache_shard_atomic(samples: List[dict], split: str, shard_id: int, base: Path = CACHE_DIR) -> Path:
    if pa is None or pa_ipc is None:
        raise RuntimeError("pyarrow is required for cache shards")
    d = split_cache_dir(split, base)
    final = d / f"{split}_shard_{shard_id:06d}.arrow"
    if final.exists() and not FORCE_REBUILD_CACHE:
        return final
    tmp = final.with_suffix(final.suffix + ".tmp")
    rows = [sample_to_cache_row(s) for s in samples]
    table = pa.Table.from_pylist(rows)
    with pa.OSFile(str(tmp), "wb") as sink:
        with pa_ipc.new_file(sink, table.schema) as writer:
            writer.write_table(table)
    os.replace(tmp, final)
    LOGGERS["cache"].info("wrote shard split=%s path=%s rows=%d", split, final, len(samples))
    return final

def iter_cache_shards(split: str, base: Path = CACHE_DIR) -> List[Path]:
    d = split_cache_dir(split, base)
    return sorted(p for p in d.glob(f"{split}_shard_*.arrow") if not p.name.endswith(".tmp"))

def read_cache_shard(path: Path) -> List[dict]:
    if pa_ipc is None:
        raise RuntimeError("pyarrow is required for cache shards")
    with pa.memory_map(str(path), "r") as source:
        reader = pa_ipc.open_file(source)
        table = reader.read_all()
    return table.to_pylist()

def rebuild_cache_index(split: str, base: Path = CACHE_DIR) -> Dict[str, dict]:
    index = {}
    for shard_path in iter_cache_shards(split, base):
        try:
            rows = read_cache_shard(shard_path)
            for i, row in enumerate(rows):
                index[row["uid"]] = {"uid": row["uid"], "shard_path": str(shard_path), "row_idx": i}
        except Exception as exc:
            log_exception(f"rebuild_cache_index:{shard_path}", exc)
    path = cache_index_path(split, base)
    jsonl_write(path, index.values())
    LOGGERS["cache"].info("rebuilt cache index split=%s entries=%d", split, len(index))
    return index

def load_cache_index(split: str, base: Path = CACHE_DIR) -> Dict[str, dict]:
    path = cache_index_path(split, base)
    if not path.exists():
        return rebuild_cache_index(split, base)
    try:
        rows = jsonl_read(path)
        return {r["uid"]: r for r in rows}
    except Exception as exc:
        log_exception(f"load_cache_index:{path}", exc)
        return rebuild_cache_index(split, base)

def append_cache_index(split: str, entries: List[dict], base: Path = CACHE_DIR) -> None:
    path = cache_index_path(split, base)
    existing = load_cache_index(split, base)
    for e in entries:
        existing[e["uid"]] = e
    jsonl_write(path, existing.values())

def load_cached_sample(entry: dict) -> dict:
    state = worker_state()["cache_shards"]
    path = entry["shard_path"]
    if path not in state:
        state[path] = read_cache_shard(Path(path))
    return cache_row_to_sample(state[path][int(entry["row_idx"])])

def clean_tmp_cache_files(base: Path = CACHE_DIR) -> None:
    for tmp in Path(base).rglob("*.tmp"):
        LOGGERS["cache"].warning("Ignoring incomplete tmp cache file: %s", tmp)

clean_tmp_cache_files(CACHE_DIR)
print("Cache utilities are ready.")


## 14. Lazy Dataset with Cache Hit/Miss Logic


In [ ]:
class QwenManifestDataset(Dataset):
    def __init__(self, rows: List[dict], processor, split: str, cache_base: Path = CACHE_DIR):
        self.rows = list(rows)
        self.processor = processor
        self.split = split
        self.cache_base = Path(cache_base)
        self.index = load_cache_index(split, self.cache_base)
        self.failed = collections.Counter()
        self._last_index_refresh = 0.0
        self._misses_since_refresh = 0

    def __len__(self):
        return len(self.rows)

    def refresh_index(self, force: bool = False):
        now = time.time()
        if force or now - self._last_index_refresh > 30.0 or self._misses_since_refresh >= 128:
            self.index = load_cache_index(self.split, self.cache_base)
            self._last_index_refresh = now
            self._misses_since_refresh = 0

    def __getitem__(self, idx: int) -> dict:
        row = self.rows[idx]
        uid = row["uid"]
        entry = self.index.get(uid)
        if not entry:
            self._misses_since_refresh += 1
            self.refresh_index()
            entry = self.index.get(uid)
        if entry:
            try:
                CACHE_STATS["hit"] += 1
                return load_cached_sample(entry)
            except Exception as exc:
                log_exception("load_cached_sample", exc, row)
        try:
            CACHE_STATS["miss"] += 1
            return process_manifest_row(row, self.processor)
        except Exception as exc:
            self.failed[uid] += 1
            log_exception("dataset_getitem", exc, row)
            # Deterministic replacement keeps DataLoader alive outside DEBUG_STRICT.
            if len(self.rows) <= 1:
                raise
            return self.__getitem__((idx + 1) % len(self.rows))

print("Lazy dataset is ready.")


## 15. Background Cache Builder

This starts in a daemon thread, uses a bounded CPU worker pool, writes shard files atomically, and never blocks the trainer on cache completion.


In [ ]:
class BackgroundCacheBuilder:
    def __init__(self, rows_by_split: Dict[str, List[dict]], processor, cache_base: Path = CACHE_DIR, shard_size: int = CACHE_SHARD_SIZE, num_workers: int = CACHE_NUM_WORKERS):
        self.rows_by_split = rows_by_split
        self.processor = processor
        self.cache_base = Path(cache_base)
        self.shard_size = int(shard_size)
        self.num_workers = max(1, int(num_workers))
        self.stop_event = threading.Event()
        self.thread = None
        self.done = threading.Event()
        self.errors = []

    def start(self):
        if self.thread and self.thread.is_alive():
            return self
        self.thread = threading.Thread(target=self._run, name="qwen-cache-builder", daemon=True)
        self.thread.start()
        LOGGERS["cache"].info("background cache builder started workers=%d", self.num_workers)
        print("Background cache builder started.")
        return self

    def stop(self, timeout: Optional[float] = 5.0):
        self.stop_event.set()
        if self.thread:
            self.thread.join(timeout=timeout)
        LOGGERS["cache"].info("background cache builder stop requested")

    def _next_shard_id(self, split: str) -> int:
        existing = iter_cache_shards(split, self.cache_base)
        if not existing:
            return 0
        pat = re.compile(r"_(\d{6})\.arrow$")
        nums = [int(m.group(1)) for p in existing if (m := pat.search(p.name))]
        return max(nums, default=-1) + 1

    def _missing_rows(self, split: str, rows: List[dict]) -> List[dict]:
        index = load_cache_index(split, self.cache_base)
        return [r for r in rows if r["uid"] not in index]

    def _process_one(self, row: dict) -> Optional[dict]:
        if self.stop_event.is_set():
            return None
        try:
            return process_manifest_row(row, self.processor)
        except Exception as exc:
            log_exception("background_cache_process", exc, row)
            return None

    def _run_split(self, split: str, rows: List[dict]):
        missing = self._missing_rows(split, rows)
        LOGGERS["cache"].info("split=%s cache missing=%d total=%d", split, len(missing), len(rows))
        if not missing:
            return
        shard = []
        shard_id = self._next_shard_id(split)
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.num_workers, thread_name_prefix=f"cache-{split}") as ex:
            futures = [ex.submit(self._process_one, row) for row in missing]
            for fut in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc=f"cache {split}"):
                if self.stop_event.is_set():
                    break
                sample = fut.result()
                if sample is None:
                    continue
                shard.append(sample)
                if len(shard) >= self.shard_size:
                    path = write_cache_shard_atomic(shard, split, shard_id, self.cache_base)
                    append_cache_index(split, [{"uid": s["uid"], "shard_path": str(path), "row_idx": i} for i, s in enumerate(shard)], self.cache_base)
                    shard_id += 1
                    shard = []
            if shard and not self.stop_event.is_set():
                path = write_cache_shard_atomic(shard, split, shard_id, self.cache_base)
                append_cache_index(split, [{"uid": s["uid"], "shard_path": str(path), "row_idx": i} for i, s in enumerate(shard)], self.cache_base)

    def _run(self):
        try:
            with contextlib.suppress(Exception):
                os.nice(5)
            for split in ["train", "val", "test"]:
                if self.stop_event.is_set():
                    break
                self._run_split(split, self.rows_by_split.get(split, []))
        except Exception as exc:
            self.errors.append(exc)
            log_exception("background_cache_run", exc)
        finally:
            self.done.set()
            LOGGERS["cache"].info("background cache builder finished errors=%d", len(self.errors))

print("Background cache builder is ready.")


## 16. Mixed Source Sampler / Batch Mixer

The custom Trainer below uses this batch sampler for training so each batch pulls from both train source roots when possible.


In [ ]:
class BalancedSourceBatchSampler(BatchSampler):
    def __init__(self, rows: List[dict], batch_size: int, seed: int = SPLIT_SEED, drop_last: bool = False):
        self.rows = rows
        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.drop_last = drop_last
        self.by_group = collections.defaultdict(list)
        for idx, row in enumerate(rows):
            self.by_group[row.get("source_group", "unknown")].append(idx)
        self.groups = sorted(self.by_group)

    def __iter__(self):
        rng = random.Random(self.seed)
        pools = {g: list(v) for g, v in self.by_group.items()}
        for g in pools:
            rng.shuffle(pools[g])
        ptr = {g: 0 for g in pools}
        active = [g for g in self.groups if pools[g]]
        batch_num = 0
        while active:
            batch = []
            if BALANCE_SOURCES_IN_BATCH and len(active) > 1:
                while len(batch) < self.batch_size and active:
                    for g in list(active):
                        if len(batch) >= self.batch_size:
                            break
                        if ptr[g] >= len(pools[g]):
                            active.remove(g)
                            LOGGERS["train"].info("source group exhausted: %s", g)
                            continue
                        batch.append(pools[g][ptr[g]])
                        ptr[g] += 1
            else:
                g = active[batch_num % len(active)]
                while len(batch) < self.batch_size and active:
                    if ptr[g] >= len(pools[g]):
                        active.remove(g)
                        if not active:
                            break
                        g = active[batch_num % len(active)]
                        continue
                    batch.append(pools[g][ptr[g]])
                    ptr[g] += 1
            if batch and (len(batch) == self.batch_size or not self.drop_last):
                if batch_num < 20:
                    dist = collections.Counter(self.rows[i].get("source_group", "unknown") for i in batch)
                    LOGGERS["train"].info("batch %d source distribution %s", batch_num, dict(dist))
                    print("batch", batch_num, dict(dist))
                batch_num += 1
                yield batch

    def __len__(self):
        n = len(self.rows) // self.batch_size
        return n if self.drop_last or len(self.rows) % self.batch_size == 0 else n + 1

class CastFloatInputsQwenTrainer(Trainer):
    def _prepare_inputs(self, inputs):
        meta_keys = ["uids", "source_groups"]
        meta = {k: inputs.pop(k) for k in list(inputs.keys()) if k in meta_keys}
        inputs = super()._prepare_inputs(inputs)
        model_dtype = getattr(self.model, "dtype", None)
        if model_dtype is not None:
            for k, v in list(inputs.items()):
                if torch.is_tensor(v) and v.is_floating_point():
                    inputs[k] = v.to(dtype=model_dtype)
        inputs.update(meta)
        return inputs

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        inputs = {k: v for k, v in inputs.items() if k not in {"uids", "source_groups"}}
        return super().compute_loss(model, inputs, return_outputs=return_outputs, **kwargs)

    def get_train_dataloader(self):
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")
        if MIX_TRAIN_SOURCES and isinstance(self.train_dataset, QwenManifestDataset):
            sampler = BalancedSourceBatchSampler(self.train_dataset.rows, self.args.train_batch_size, SPLIT_SEED)
            return DataLoader(
                self.train_dataset,
                batch_sampler=sampler,
                collate_fn=self.data_collator,
                num_workers=self.args.dataloader_num_workers,
                pin_memory=self.args.dataloader_pin_memory,
                persistent_workers=self.args.dataloader_persistent_workers if self.args.dataloader_num_workers > 0 else False,
                prefetch_factor=self.args.dataloader_prefetch_factor if self.args.dataloader_num_workers > 0 else None,
            )
        return super().get_train_dataloader()

print("Mixed sampler and Trainer subclass are ready.")


## 17. Metrics: WER, CER, and Prediction Tables


In [ ]:
def edit_distance(a: Sequence[Any], b: Sequence[Any]) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb))
        prev = cur
    return prev[-1]

def wer(ref: str, hyp: str, *, loose: bool = False, punctuation_insensitive: bool = False) -> float:
    r = normalize_metric_text(ref, loose=loose, punctuation_insensitive=punctuation_insensitive).split()
    h = normalize_metric_text(hyp, loose=loose, punctuation_insensitive=punctuation_insensitive).split()
    return edit_distance(r, h) / max(1, len(r))

def cer(ref: str, hyp: str, *, loose: bool = False, punctuation_insensitive: bool = False) -> float:
    r = list(normalize_metric_text(ref, loose=loose, punctuation_insensitive=punctuation_insensitive))
    h = list(normalize_metric_text(hyp, loose=loose, punctuation_insensitive=punctuation_insensitive))
    return edit_distance(r, h) / max(1, len(r))

def model_device() -> torch.device:
    return next(model.parameters()).device

@contextlib.contextmanager
def maybe_disable_adapters(disable: bool = False):
    if disable and hasattr(model, "disable_adapter"):
        with model.disable_adapter():
            yield
    else:
        yield

@torch.no_grad()
def decode_qwen_generate_output(out) -> str:
    """Qwen3-ASR generate may return token ids or already-decoded strings depending on backend."""
    if isinstance(out, str):
        return out
    if isinstance(out, (list, tuple)) and out and isinstance(out[0], str):
        return out[0]
    if torch.is_tensor(out):
        return processor.tokenizer.batch_decode(out, skip_special_tokens=True)[0]
    if isinstance(out, (list, tuple)) and out and torch.is_tensor(out[0]):
        return processor.tokenizer.batch_decode(out, skip_special_tokens=True)[0]
    return str(out)

@torch.no_grad()
def transcribe_audio_array(audio: np.ndarray, sr: int = SAMPLE_RATE, language: Optional[str] = "Arabic", *, disable_adapters: bool = False) -> str:
    model.eval()
    audio = resample_audio(audio, sr, SAMPLE_RATE)

    # For baseline predictions, use the official qwen_asr wrapper. It handles chunking,
    # prompt format, backend string outputs, and language parsing correctly.
    if disable_adapters:
        try:
            results = asr_wrapper.transcribe(audio=(audio, SAMPLE_RATE), language=language, return_time_stamps=False)
            return normalize_arabic_text(results[0].text if results else "")
        except Exception as wrapper_exc:
            log_exception("transcribe_audio_array_wrapper_baseline", wrapper_exc)

    messages = build_prefix_messages("", audio)
    try:
        prefix_text = processor.apply_chat_template([messages], add_generation_prompt=True, tokenize=False)
        prefix_text = prefix_text[0] if isinstance(prefix_text, list) else prefix_text
        inputs = processor(text=[prefix_text], audio=[audio], return_tensors="pt", padding=True)
        inputs = {k: (v.to(model_device()) if torch.is_tensor(v) else v) for k, v in inputs.items()}
        for k, v in list(inputs.items()):
            if torch.is_tensor(v) and v.is_floating_point() and getattr(model, "dtype", None) is not None:
                inputs[k] = v.to(dtype=getattr(model, "dtype"))
        with maybe_disable_adapters(disable_adapters):
            out = model.generate(**inputs, max_new_tokens=256)
        decoded = decode_qwen_generate_output(out)
        if "<asr_text>" in decoded:
            decoded = decoded.split("<asr_text>")[-1]
        return normalize_arabic_text(decoded)
    except Exception as exc:
        log_exception("transcribe_audio_array_generate", exc)
        try:
            results = asr_wrapper.transcribe(audio=(audio, SAMPLE_RATE), language=language, return_time_stamps=False)
            return normalize_arabic_text(results[0].text if results else "")
        except Exception as wrapper_exc:
            log_exception("transcribe_audio_array_wrapper", wrapper_exc)
        return ""

def prediction_record(row: dict, pred: str) -> dict:
    ref = row.get("text", "")
    return {
        "uid": row["uid"], "source": row.get("source"), "source_group": row.get("source_group"), "split": row.get("split"),
        "duration": row.get("duration"), "reference": ref, "prediction": pred,
        "normalized_reference": normalize_metric_text(ref),
        "normalized_prediction": normalize_metric_text(pred),
        "wer": wer(ref, pred), "cer": cer(ref, pred),
        "wer_loose": wer(ref, pred, loose=True), "cer_loose": cer(ref, pred, loose=True),
        "wer_no_punct": wer(ref, pred, punctuation_insensitive=True), "cer_no_punct": cer(ref, pred, punctuation_insensitive=True),
    }

def save_prediction_table(rows: List[dict], predictions: List[str], name: str) -> Path:
    table = [prediction_record(row, pred) for row, pred in zip(rows, predictions)]
    path = PREDICTION_DIR / f"{name}.jsonl"
    jsonl_write(path, table)
    LOGGERS["eval"].info("saved predictions path=%s rows=%d", path, len(table))
    return path

def summarize_prediction_records(records: List[dict]) -> dict:
    if not records:
        return {"num_predictions": 0}
    keys = ["wer", "cer", "wer_loose", "cer_loose", "wer_no_punct", "cer_no_punct"]
    out = {"num_predictions": len(records)}
    for key in keys:
        vals = [float(r[key]) for r in records if r.get(key) is not None and math.isfinite(float(r[key]))]
        out[key] = statistics.mean(vals) if vals else None
    out["total_hours"] = sum(float(r.get("duration") or 0.0) for r in records) / 3600.0
    out["by_source_group"] = {}
    for group, group_records in collections.defaultdict(list, {g: [r for r in records if r.get("source_group") == g] for g in sorted({r.get("source_group") for r in records})}).items():
        out["by_source_group"][group] = {"rows": len(group_records), "wer": statistics.mean([float(r["wer"]) for r in group_records]) if group_records else None, "cer": statistics.mean([float(r["cer"]) for r in group_records]) if group_records else None}
    return out

@torch.no_grad()
def run_generation_eval(rows: List[dict], max_samples: int = 16, name: str = "eval_preview", *, disable_adapters: bool = False) -> dict:
    rows = rows[:max_samples]
    predictions = []
    for row in tqdm(rows, desc="generation eval"):
        try:
            audio, sr = load_audio_for_row(row)
            predictions.append(transcribe_audio_array(audio, sr, language="Arabic", disable_adapters=disable_adapters))
        except Exception as exc:
            log_exception("run_generation_eval", exc, row)
            predictions.append("")
    path = save_prediction_table(rows, predictions, name)
    records = jsonl_read(path)
    metrics = summarize_prediction_records(records)
    metrics["prediction_path"] = str(path)
    print(metrics)
    return metrics

def run_resumable_test_predictions(rows: List[dict], path: Path, metrics_path: Path, *, disable_adapters: bool, force: bool = False, max_samples: Optional[int] = None, label: str = "baseline") -> dict:
    selected = list(rows if max_samples is None else rows[:int(max_samples)])
    path.parent.mkdir(parents=True, exist_ok=True)
    if force and path.exists():
        path.unlink()
    existing = {}
    if path.exists():
        for rec in jsonl_read(path):
            # Treat empty predictions as missing so a failed baseline run is self-healing.
            if str(rec.get("prediction", "")).strip():
                existing[rec["uid"]] = rec
    selected_by_uid = {r["uid"]: r for r in selected}
    records = [existing[uid] for uid in selected_by_uid if uid in existing]
    missing = [r for r in selected if r["uid"] not in existing]
    print(f"{label}: selected={len(selected)} usable_existing={len(existing)} missing={len(missing)} path={path}")
    # Rewrite the file when any rows are missing, preserving only usable existing predictions.
    mode = "w" if missing else "a"
    if mode == "w":
        with path.open("w", encoding="utf-8") as f:
            for uid in selected_by_uid:
                if uid in existing:
                    f.write(json.dumps(existing[uid], ensure_ascii=False, sort_keys=True) + "\n")
    with path.open(mode, encoding="utf-8") as f:
        for row in tqdm(missing, desc=f"{label} test predictions"):
            try:
                audio, sr = load_audio_for_row(row)
                pred = transcribe_audio_array(audio, sr, language=BASELINE_GENERATION_LANGUAGE, disable_adapters=disable_adapters)
                rec = prediction_record(row, pred)
            except Exception as exc:
                log_exception(f"{label}_test_prediction", exc, row)
                rec = prediction_record(row, "")
                rec["error"] = repr(exc)
            f.write(json.dumps(rec, ensure_ascii=False, sort_keys=True) + "\n")
            f.flush()
            records.append(rec)
    all_records_by_uid = {r["uid"]: r for r in jsonl_read(path)}
    ordered_records = [all_records_by_uid[uid] for uid in selected_by_uid if uid in all_records_by_uid]
    metrics = summarize_prediction_records(ordered_records)
    metrics.update({"prediction_path": str(path), "label": label, "max_samples": max_samples, "disable_adapters": disable_adapters})
    metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print(json.dumps(metrics, ensure_ascii=False, indent=2, default=str))
    LOGGERS["eval"].info("%s metrics: %s", label, metrics)
    return metrics

print("Metric and prediction helpers are ready.")


## 18. Smoke Test with Fake Train/Val/Test Samples

This validates normalization, lazy loading, cache shard writing, cache index loading, collation, a model forward pass, a tiny optimizer step, and generation.


In [ ]:
def make_smoke_manifest_rows() -> List[dict]:
    rng = np.random.default_rng(123)
    examples = [
        ("smoke_train", "train", "مرحبا هذا اختبار", 1.2),
        ("smoke_val", "val", "السلام عليكم", 1.4),
        ("smoke_test", "test", "كيف الحال", 1.6),
    ]
    rows = []
    for uid, split, text, seconds in examples:
        t = np.linspace(0, seconds, int(SAMPLE_RATE * seconds), endpoint=False)
        audio = 0.05 * np.sin(2 * np.pi * 220 * t) + 0.005 * rng.standard_normal(len(t))
        rows.append(base_manifest_row(
            uid=uid,
            source="smoke",
            source_root=RUN_DIR,
            source_group=f"smoke_{split}",
            original_split=split,
            split=split,
            text=text,
            duration=seconds,
            audio_kind="array",
            audio_array=audio.astype(np.float32).tolist(),
            sampling_rate=SAMPLE_RATE,
        ))
    jsonl_write(MANIFEST_SMOKE_PATH, rows)
    return rows

def run_smoke_test():
    rows = make_smoke_manifest_rows()
    print("1. fake manifest rows created")
    for r in rows:
        print(r["uid"], normalize_arabic_text(r["text"]))
    print("2. Arabic normalization ok")
    rows_by_split = {s: [r for r in rows if r["split"] == s] for s in ["train", "val", "test"]}
    builder = BackgroundCacheBuilder(rows_by_split, processor, cache_base=SMOKE_CACHE_DIR, shard_size=1, num_workers=1).start()
    builder.thread.join(timeout=300)
    assert builder.done.is_set(), "smoke cache builder did not finish"
    print("3. smoke cache builder finished")
    for split in ["train", "val", "test"]:
        assert iter_cache_shards(split, SMOKE_CACHE_DIR), f"no smoke shard for {split}"
        assert load_cache_index(split, SMOKE_CACHE_DIR), f"no smoke index for {split}"
    print("4. cache shards and indexes exist")
    ds = QwenManifestDataset(rows_by_split["train"], processor, "train", cache_base=SMOKE_CACHE_DIR)
    sample = ds[0]
    assert "input_ids" in sample and "labels" in sample
    print("5. cached sample loaded")
    batch = collator([sample])
    print("6. collator batch shapes", {k: tuple(v.shape) for k, v in batch.items() if torch.is_tensor(v)})
    model.train()
    device_batch = {k: v.to(model.device) for k, v in batch.items() if torch.is_tensor(v)}
    for k, v in list(device_batch.items()):
        if v.is_floating_point():
            device_batch[k] = v.to(dtype=getattr(model, "dtype", MODEL_DTYPE))
    out = model(**device_batch)
    assert getattr(out, "loss", None) is not None
    print("7. forward pass loss", float(out.loss.detach().cpu()))
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-6)
    out.loss.backward()
    opt.step(); opt.zero_grad(set_to_none=True)
    print("8. one tiny train step ok")
    pred = transcribe_audio_array(np.asarray(rows[1]["audio_array"], dtype=np.float32), SAMPLE_RATE, language="Arabic")
    print("9. generation/transcription smoke output:", pred[:120])
    print("SMOKE TEST SUCCESS")

if RUN_SMOKE_TEST:
    run_smoke_test()
    if SMOKE_TEST_ONLY:
        raise SystemExit("SMOKE_TEST_ONLY=True; stopping after smoke test.")
else:
    print("RUN_SMOKE_TEST=False; skipped.")


## 19. Small Real Sample Test

This scans actual roots, prints Arrow/Parquet schemas, parses QASR if available, builds a mini manifest, applies split rules, loads/processes one sample per source, and collates a mixed batch.


In [ ]:
def small_real_sample_test() -> List[dict]:
    mini = []
    for sr in SOURCE_ROOTS:
        limits = {"arrow": MAX_OMNILINGUAL_SAMPLES_DEBUG, "parquet": MAX_CASABLANCA_SAMPLES_DEBUG, "qasr_segments": MAX_QASR_SEGMENTS_DEBUG, "arrow_total": MAX_OMNILINGUAL_SAMPLES_DEBUG, "parquet_total": MAX_CASABLANCA_SAMPLES_DEBUG, "qasr_total": MAX_QASR_SEGMENTS_DEBUG}
        got = build_manifest_rows_for_root(sr, limits=limits)
        mini.extend(got[: max(MAX_OMNILINGUAL_SAMPLES_DEBUG, MAX_CASABLANCA_SAMPLES_DEBUG, MAX_QASR_SEGMENTS_DEBUG)])
    # Optional QASR direct example/root diagnostic even when QASR is not part of full training sources.
    if QASR_ROOT.exists() and not any(r.get("source") == "qasr" for r in mini):
        qsr = SourceRoot(QASR_ROOT, "train", "qasr")
        got = build_manifest_rows_for_root(qsr, limits={"qasr_segments": MAX_QASR_SEGMENTS_DEBUG, "qasr_total": MAX_QASR_SEGMENTS_DEBUG})
        mini.extend(got[:MAX_QASR_SEGMENTS_DEBUG])
    mini = assign_splits(mini)
    log_manifest_counts(mini, "small real sample")
    processed = []
    seen_sources = set()
    for row in mini:
        if row["source"] in seen_sources:
            continue
        try:
            audio, sr = load_audio_for_row(row)
            sample = process_manifest_row(row, processor)
            processed.append(sample)
            seen_sources.add(row["source"])
            print({"uid": row["uid"], "source": row["source"], "source_group": row["source_group"], "split": row["split"], "duration": row.get("duration"), "text": row.get("text", "")[:80], "audio_shape": audio.shape, "sr": sr})
        except Exception as exc:
            log_exception("small_real_sample_process", exc, row)
    if processed:
        b = collator(processed)
        print("small real collator shapes", {k: tuple(v.shape) for k, v in b.items() if torch.is_tensor(v)})
    print("SMALL REAL SAMPLE TEST COMPLETE")
    return mini

if RUN_SMALL_REAL_SAMPLE_TEST:
    SMALL_REAL_ROWS = small_real_sample_test()
else:
    SMALL_REAL_ROWS = []
    print("RUN_SMALL_REAL_SAMPLE_TEST=False; skipped.")


## 20. Build or Reuse Full Manifests


In [ ]:
def row_duration_seconds(row: dict) -> float:
    try:
        return float(row.get("duration") or 0.0)
    except Exception:
        return 0.0

def hours(rows: List[dict]) -> float:
    return sum(row_duration_seconds(r) for r in rows) / 3600.0

def stable_row_order(rows: List[dict], salt: str) -> List[dict]:
    return sorted(rows, key=lambda r: stable_hash(f"{salt}|{r.get('uid')}", SPLIT_SEED))

def select_rows_up_to_hours(rows: List[dict], target_hours: float, *, salt: str) -> List[dict]:
    selected = []
    total = 0.0
    for row in stable_row_order(rows, salt):
        dur = row_duration_seconds(row)
        if dur <= 0:
            continue
        if selected and total + dur > target_hours * 3600.0:
            continue
        selected.append(row)
        total += dur
        if total >= target_hours * 3600.0:
            break
    return selected

def duration_desc_order(rows: List[dict], salt: str) -> List[dict]:
    return sorted(rows, key=lambda r: (-row_duration_seconds(r), stable_hash(f"{salt}|{r.get('uid')}", SPLIT_SEED)))

def duration_asc_order(rows: List[dict], salt: str) -> List[dict]:
    return sorted(rows, key=lambda r: (row_duration_seconds(r), stable_hash(f"{salt}|{r.get('uid')}", SPLIT_SEED)))

def select_train_subset_by_hours(rows: List[dict], target_hours: float) -> List[dict]:
    target_seconds = float(target_hours) * 3600.0
    selected = []
    selected_uids = set()
    total = 0.0

    # Keep a small Casablanca slice so the mini run still covers both train roots,
    # then fill the 50h cap with longest clips to avoid a 10k-row preprocessing run.
    casa_rows = [r for r in stable_row_order(rows, "mini-casablanca-seed") if r.get("source_group") == "casablanca_relevant_arabic"]
    for row in casa_rows[: int(MINI_CASABLANCA_MIN_ROWS)]:
        dur = row_duration_seconds(row)
        if dur <= 0 or total + dur > target_seconds:
            continue
        selected.append(row)
        selected_uids.add(row["uid"])
        total += dur

    for row in duration_desc_order([r for r in rows if r["uid"] not in selected_uids], "mini-longest"):
        dur = row_duration_seconds(row)
        if dur <= 0:
            continue
        if total + dur > target_seconds:
            continue
        selected.append(row)
        selected_uids.add(row["uid"])
        total += dur
        if total >= target_seconds:
            break

    # Fill small gaps with short rows without exceeding 50h.
    for row in duration_asc_order([r for r in rows if r["uid"] not in selected_uids], "mini-topoff-short"):
        dur = row_duration_seconds(row)
        if dur <= 0:
            continue
        if total + dur > target_seconds:
            continue
        selected.append(row)
        selected_uids.add(row["uid"])
        total += dur
        if total >= target_seconds:
            break

    return stable_row_order(selected, "mini-train-final-2")

def select_eval_rows(rows: List[dict], max_samples: int, *, salt: str) -> List[dict]:
    return stable_row_order(rows, salt)[: int(max_samples)]

ALL_ROWS = build_or_load_full_manifest()
FULL_TRAIN_ROWS = [r for r in ALL_ROWS if r.get("split") == "train"]
FULL_VAL_ROWS = [r for r in ALL_ROWS if r.get("split") == "val"]
FULL_TEST_ROWS = [r for r in ALL_ROWS if r.get("split") == "test"]

TRAIN_ROWS = select_train_subset_by_hours(FULL_TRAIN_ROWS, MINI_MAX_TRAIN_HOURS)
VAL_ROWS = select_eval_rows(FULL_VAL_ROWS, MINI_MAX_EVAL_SAMPLES, salt="mini-val-100")
TEST_ROWS = select_eval_rows(FULL_TEST_ROWS, MINI_MAX_EVAL_SAMPLES, salt="mini-test-100")
ROWS_BY_SPLIT = {"train": TRAIN_ROWS, "val": VAL_ROWS, "test": TEST_ROWS}

selection_summary = {
    "run_id": MINI_RUN_ID,
    "full_counts": {"train": len(FULL_TRAIN_ROWS), "val": len(FULL_VAL_ROWS), "test": len(FULL_TEST_ROWS)},
    "selected_counts": {k: len(v) for k, v in ROWS_BY_SPLIT.items()},
    "selected_hours": {k: hours(v) for k, v in ROWS_BY_SPLIT.items()},
    "train_hours_cap": MINI_MAX_TRAIN_HOURS,
    "eval_sample_cap": MINI_MAX_EVAL_SAMPLES,
    "train_by_source_group": {
        g: {"rows": len(xs), "hours": hours(xs)}
        for g, xs in sorted(collections.defaultdict(list, {g: [r for r in TRAIN_ROWS if r.get("source_group") == g] for g in sorted({r.get("source_group") for r in TRAIN_ROWS})}).items())
    },
}
print(json.dumps(selection_summary, ensure_ascii=False, indent=2, default=str))
(MANIFEST_DIR / "mini_selection_summary.json").write_text(json.dumps(selection_summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
jsonl_write(MANIFEST_DIR / "train" / "manifest_train_mini_50h.jsonl", TRAIN_ROWS)
jsonl_write(MANIFEST_DIR / "val" / "manifest_val_mini_100.jsonl", VAL_ROWS)
jsonl_write(MANIFEST_DIR / "test" / "manifest_test_mini_100.jsonl", TEST_ROWS)

assert TRAIN_ROWS, "No training rows found. Check TRAIN_SOURCE_DIRS and manifest parser logs."
assert VAL_ROWS and TEST_ROWS, "Validation/test rows are required from held-out roots."
assert hours(TRAIN_ROWS) >= MINI_MIN_TRAIN_HOURS_REQUIRED, f"Mini train subset is only {hours(TRAIN_ROWS):.2f}h; requested about {MINI_MAX_TRAIN_HOURS:.2f}h."
assert len(VAL_ROWS) <= MINI_MAX_EVAL_SAMPLES and len(TEST_ROWS) <= MINI_MAX_EVAL_SAMPLES


## 21. Baseline Test Predictions and Evaluation

Before training, generate baseline predictions on the held-out test set with LoRA adapters disabled. The JSONL file is resumable: if a previous run exists and `FORCE_REGENERATE_BASELINE=False`, only missing UIDs are generated.


In [ ]:
baseline_test_metrics = run_resumable_test_predictions(
    TEST_ROWS,
    BASELINE_TEST_PREDICTIONS_PATH,
    BASELINE_TEST_METRICS_PATH,
    disable_adapters=True,
    force=FORCE_REGENERATE_BASELINE,
    max_samples=MAX_BASELINE_TEST_SAMPLES,
    label="mini_50h_100eval_baseline_test",
)
print("Mini baseline test metrics saved to", BASELINE_TEST_METRICS_PATH)
print("Mini baseline test predictions saved to", BASELINE_TEST_PREDICTIONS_PATH)


## 22. Start Background Cache Builder

Training can begin immediately after this cell; cache hits will increase over time as complete shards appear.


In [ ]:
train_dataset = QwenManifestDataset(TRAIN_ROWS, processor, "train", CACHE_DIR)
eval_dataset = QwenManifestDataset(VAL_ROWS, processor, "val", CACHE_DIR)
test_dataset = QwenManifestDataset(TEST_ROWS, processor, "test", CACHE_DIR)
print("Datasets ready:", len(train_dataset), len(eval_dataset), len(test_dataset))
print("Initial cache indexes:", {s: len(load_cache_index(s, CACHE_DIR)) for s in ["train", "val", "test"]})

# Start this after dataset initialization so cache-index bootstrap cannot race on the same .tmp file.
cache_builder = BackgroundCacheBuilder(ROWS_BY_SPLIT, processor, CACHE_DIR, CACHE_SHARD_SIZE, CACHE_NUM_WORKERS).start()


## 23. TrainingArguments, Trainer, and Timed Save/Eval Callback


In [ ]:
def find_latest_checkpoint(output_dir: Path) -> Optional[str]:
    if not output_dir.exists():
        return None
    best = None
    for p in output_dir.iterdir():
        m = re.match(r"checkpoint-(\d+)$", p.name)
        if p.is_dir() and m:
            step = int(m.group(1))
            if best is None or step > best[0]:
                best = (step, str(p))
    return best[1] if best else None

class TimedEvalSaveCallback(TrainerCallback):
    def __init__(self, every_seconds: float = 3600.0):
        self.every_seconds = every_seconds
        self.last = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        now = time.time()
        if now - self.last >= self.every_seconds:
            control.should_evaluate = True
            control.should_save = True
            self.last = now
            LOGGERS["train"].info("Timed eval/save triggered at global_step=%s", state.global_step)
        return control

class BestConfigCallback(TrainerCallback):
    def __init__(self, best_dir: Path):
        self.best_dir = Path(best_dir)
        self.best_dir.mkdir(parents=True, exist_ok=True)
        self.best_path = self.best_dir / "best_config.json"
        self.best_metric = math.inf
        if self.best_path.exists():
            with contextlib.suppress(Exception):
                self.best_metric = json.loads(self.best_path.read_text()).get("best_metric", math.inf)

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        metrics = metrics or {}
        metric = metrics.get("eval_wer", metrics.get("eval_loss", math.inf))
        if metric is None:
            return control
        if float(metric) < float(self.best_metric):
            self.best_metric = float(metric)
            payload = {
                "best_metric": self.best_metric,
                "global_step": state.global_step,
                "model_name": MODEL_NAME,
                "run_dir": str(RUN_DIR),
                "lora_config": getattr(lora_config, "to_dict", lambda: str(lora_config))(),
                "data_counts": {k: len(v) for k, v in ROWS_BY_SPLIT.items()},
                "cache_stats": dict(CACHE_STATS),
                "config": CONFIG,
            }
            self.best_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
            with contextlib.suppress(Exception):
                model.save_pretrained(self.best_dir)
                processor.save_pretrained(self.best_dir)
            LOGGERS["eval"].info("best model updated metric=%s path=%s", self.best_metric, self.best_dir)
        return control

def make_training_args(eval_save_steps: int = 200) -> TrainingArguments:
    return TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=MAX_GRAD_NORM,
        logging_steps=10,
        save_strategy="steps",
        save_steps=eval_save_steps,
        save_total_limit=5,
        eval_strategy="steps",
        eval_steps=eval_save_steps,
        do_eval=True,
        bf16=USE_BF16_EFFECTIVE,
        fp16=USE_FP16_EFFECTIVE and not USE_BF16_EFFECTIVE,
        dataloader_num_workers=TRAIN_DATALOADER_NUM_WORKERS,
        dataloader_pin_memory=torch.cuda.is_available(),
        dataloader_persistent_workers=PERSISTENT_WORKERS if TRAIN_DATALOADER_NUM_WORKERS > 0 else False,
        dataloader_prefetch_factor=PREFETCH_FACTOR if TRAIN_DATALOADER_NUM_WORKERS > 0 else None,
        remove_unused_columns=False,
        report_to="none",
        save_safetensors=True,
        ddp_find_unused_parameters=False,
    )

training_args = make_training_args(eval_save_steps=200)
trainer = CastFloatInputsQwenTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
    tokenizer=processor.tokenizer,
    callbacks=[TimedEvalSaveCallback(3600.0), BestConfigCallback(BEST_MODEL_DIR)],
)
print("Trainer ready.")


## 24. Estimate Eval/Save Steps for Roughly 60 Minutes

A short timed forward pass estimates steps per hour and updates `eval_steps`/`save_steps` as a backup to the wall-clock callback.


In [ ]:
def choose_eval_save_steps_after_training() -> int:
    steps_per_epoch = len(BalancedSourceBatchSampler(train_dataset.rows, TRAIN_BATCH_SIZE, SPLIT_SEED))
    steps = max(steps_per_epoch + 1, 200)
    print({"steps_per_epoch": steps_per_epoch, "EVAL_SAVE_STEPS": steps, "note": "set beyond one mini epoch to avoid extra in-training evals"})
    LOGGERS["train"].info("mini eval/save steps set beyond epoch steps_per_epoch=%s steps=%s", steps_per_epoch, steps)
    return steps

if MINI_PRECACHE_BEFORE_TRAIN:
    print("Waiting for mini cache builder to finish before training...")
    while not cache_builder.done.wait(timeout=MINI_CACHE_WAIT_POLL_SECONDS):
        counts = {s: len(load_cache_index(s, CACHE_DIR)) for s in ["train", "val", "test"]}
        print("cache progress", counts)
    if cache_builder.errors:
        raise RuntimeError(f"Background cache builder failed: {cache_builder.errors!r}")
    for ds in [train_dataset, eval_dataset, test_dataset]:
        ds.refresh_index(force=True)
    print("cache ready", {s: len(load_cache_index(s, CACHE_DIR)) for s in ["train", "val", "test"]})

EVAL_SAVE_STEPS = choose_eval_save_steps_after_training()
training_args = make_training_args(eval_save_steps=EVAL_SAVE_STEPS)
trainer = CastFloatInputsQwenTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
    tokenizer=processor.tokenizer,
    callbacks=[TimedEvalSaveCallback(10**9), BestConfigCallback(BEST_MODEL_DIR)],
)
print("Recreated trainer with final eval_steps/save_steps:", EVAL_SAVE_STEPS)


## 25. Train

This resumes from the latest checkpoint when configured. Cache construction continues in the background while training runs.


In [ ]:
resume_checkpoint = find_latest_checkpoint(OUTPUT_DIR) if RESUME_FROM_CHECKPOINT else None
print("resume_checkpoint:", resume_checkpoint)
LOGGERS["train"].info("training start resume_checkpoint=%s", resume_checkpoint)
train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
trainer.save_model(str(OUTPUT_DIR / "final_trainer_model"))
processor.save_pretrained(str(OUTPUT_DIR / "final_processor"))
LOGGERS["train"].info("training complete metrics=%s cache_stats=%s", train_result.metrics, dict(CACHE_STATS))
print(train_result.metrics)
print("cache stats", dict(CACHE_STATS))


## 26. Evaluate Best Model and Validation Preview


In [ ]:
eval_metrics = trainer.evaluate(eval_dataset=eval_dataset)
print("mini eval metrics", eval_metrics)
LOGGERS["eval"].info("mini_eval_metrics=%s", eval_metrics)
preview_metrics = run_generation_eval(VAL_ROWS, max_samples=MINI_MAX_EVAL_SAMPLES, name=f"mini_50h_100eval_val_predictions_step_{trainer.state.global_step}")
print("mini validation prediction metrics", preview_metrics)


## 27. Held-Out Test Transcription Examples


In [ ]:
test_preview_metrics = run_generation_eval(TEST_ROWS, max_samples=MINI_MAX_EVAL_SAMPLES, name=f"mini_50h_100eval_test_predictions_step_{trainer.state.global_step}")
print("mini test prediction metrics", test_preview_metrics)


## 28. Save Final Adapter, Processor, Config, and Summary


In [ ]:
def git_commit_hash() -> Optional[str]:
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(RUN_DIR.parent.parent), text=True).strip()
    except Exception:
        return None

FINAL_DIR = RUN_DIR / "final_adapter"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
summary = {
    "model_name": MODEL_NAME,
    "final_dir": str(FINAL_DIR),
    "output_dir": str(OUTPUT_DIR),
    "best_model_dir": str(BEST_MODEL_DIR),
    "manifest_all": str(MANIFEST_ALL_PATH),
    "split_assignments": {
        "val_uids": str(VAL_UIDS_PATH),
        "test_uids": str(TEST_UIDS_PATH),
        "jsonl": str(ASSIGNMENTS_JSONL_PATH),
    },
    "cache_dir": str(CACHE_DIR),
    "rows": {k: len(v) for k, v in ROWS_BY_SPLIT.items()},
    "cache_index_counts": {s: len(load_cache_index(s, CACHE_DIR)) for s in ["train", "val", "test"]},
    "cache_stats": dict(CACHE_STATS),
    "lora_config": getattr(lora_config, "to_dict", lambda: str(lora_config))(),
    "config": CONFIG,
    "environment": ENV_INFO,
    "git_commit": git_commit_hash(),
}
summary_path = RUN_DIR / "summary_report.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2, default=str))
print("Final adapter/processor/config saved.")

# Ask the background cache builder to stop after final saves. It never deletes raw data.
with contextlib.suppress(Exception):
    cache_builder.stop(timeout=5)


## 29. Mini Run Integrity Report


In [ ]:
def prediction_file_health(path: Path) -> dict:
    records = jsonl_read(path)
    empty = [r.get("uid") for r in records if not str(r.get("prediction", "")).strip()]
    return {
        "path": str(path),
        "rows": len(records),
        "empty_predictions": len(empty),
        "empty_prediction_uids": empty[:20],
        "hours": sum(float(r.get("duration") or 0.0) for r in records) / 3600.0,
    }

prediction_health = {
    "baseline_test": prediction_file_health(BASELINE_TEST_PREDICTIONS_PATH),
    "val_predictions": prediction_file_health(Path(preview_metrics["prediction_path"])),
    "test_predictions": prediction_file_health(Path(test_preview_metrics["prediction_path"])),
}

mini_results = {
    "run_id": MINI_RUN_ID,
    "notebook_path": str(NOTEBOOK_PATH),
    "run_dir": str(RUN_DIR),
    "cache_dir": str(CACHE_DIR),
    "selection_summary": selection_summary,
    "baseline_test_metrics": baseline_test_metrics,
    "trainer_eval_metrics": eval_metrics,
    "val_prediction_metrics": preview_metrics,
    "test_prediction_metrics": test_preview_metrics,
    "prediction_health": prediction_health,
    "final_adapter_dir": str(FINAL_DIR),
    "summary_report": str(summary_path),
}
mini_results_path = RUN_DIR / "mini_50h_100eval_results.json"
mini_results_path.write_text(json.dumps(mini_results, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(json.dumps(mini_results, ensure_ascii=False, indent=2, default=str))

bad = {name: h for name, h in prediction_health.items() if h["empty_predictions"] or h["rows"] == 0}
assert not bad, f"Empty or missing predictions detected: {json.dumps(bad, ensure_ascii=False, default=str)}"
assert prediction_health["baseline_test"]["rows"] <= MINI_MAX_EVAL_SAMPLES
assert prediction_health["val_predictions"]["rows"] <= MINI_MAX_EVAL_SAMPLES
assert prediction_health["test_predictions"]["rows"] <= MINI_MAX_EVAL_SAMPLES
print("MINI RUN INTEGRITY CHECK PASSED:", mini_results_path)
